# DeepLabCut Notebook with only code

In [ ]:
import deeplabcut as dlc
dlc.__version__

In [ ]:
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os

In [ ]:
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name != "deeplabcut-insect-locomotion-analysis":
    for parent in BASE_DIR.parents:
        if parent.name == "deeplabcut-insect-locomotion-analysis":
            BASE_DIR = parent
            break

# Setting the current working directory to the base directory of the project (pwd so even Path.cwd() will return the base directory)
os.chdir(BASE_DIR)

In [ ]:
pwd


In [ ]:
PROJECT_NAME: str = "Stick_Insect_Motion_Hacking_2D_Tracking"
EXPERIMENTER: str = "RrPllana"

VIDEO_DIR: str = str((BASE_DIR / "videos").resolve())
WORKING_DIR: str = str((BASE_DIR / "models").resolve())

# Dynamically create a list of video paths in the VIDEO_DIR that match the pattern "IMG_*.MOV"
video_paths = [str(path)for path in Path(VIDEO_DIR).glob("IMG_*.MOV")]
path_config_file = None

In [ ]:
path_config_file = dlc.create_new_project(PROJECT_NAME, EXPERIMENTER, video_paths, copy_videos=True)

In [ ]:
if path_config_file == 'nothingcreated' or path_config_file is None:
    CONFIG_PATH: str = str((BASE_DIR / f"{PROJECT_NAME}-{EXPERIMENTER}-2026-09-10" / "config.yaml").resolve())
else:
    CONFIG_PATH: str = path_config_file
print(f"Configuration file created at: {CONFIG_PATH}")

In [ ]:
import yaml

with open(CONFIG_PATH, 'r') as file:
    config = yaml.safe_load(file)

# Update the config dictionary with the new values from the user input without editing the config.yaml file directly
config['numframes2pick'] = 22  # Update the number of frames to pick for labeling/refinement

# Check if the 'bodyparts' and 'skeleton' keys are presented correctly in the config dictionary
print("Number of frames to pick for labeling/refinement:", config['numframes2pick'])
print("Bodyparts:", config['bodyparts'])
print("Skeleton:", config['skeleton'])

In [ ]:
%matplotlib inline
dlc.extract_frames(CONFIG_PATH, mode='automatic', algo='uniform', userfeedback=False, crop=False)

In [ ]:
%gui qt6
import napari

# If this is used, then first after opening the view, it is needed to go to Plugins -> napari-deeplabcut, to be able to proces multiple labels
# and also save the files correctly, otherwise it will not work properly.

# Next you drag and drop config file and the folder that contains the extracted frames you want to label.
napari.Viewer()

In [ ]:
# Alternative to napari.Viewer() command, it can be used to open the view with the config file and the folder that contains the extracted 
# frames you want to label.
dlc.label_frames(CONFIG_PATH)

# If you have more than one video that you wanna label (multiple folders), than run the label_frames(config file, image_folder='') for each video data folder
# dlc.label_frames(CONFIG_PATH, image_folder='/home/yuchen/deeplab-insect-locomotion-analysis/Stick_Insect_Motion_Hacking_2D_Tracking-RrPllana-2026-09-10/labeled-data/IMG_6948_2seconds_protractor') 

In [ ]:
dlc.check_labels(CONFIG_PATH, draw_skeleton=True)

In [ ]:
dlc.create_training_dataset(CONFIG_PATH)

In [ ]:
# Running with GPU and specifying the network type as "resnet_50" and engine as "pytorch"
dlc.create_training_dataset(CONFIG_PATH, net_type="resnet_50", engine=dlc.Engine.PYTORCH)

In [ ]:
dlc.train_network(
    CONFIG_PATH,   # path to the config.yaml file
    shuffle=8,     # shuffle the training data
    save_epochs=20, # save the model every 20 epochs
    epochs=2000,    # number of training epochs
    batch_size=8,  # batch size for training (ch anging this number changes how we utilize the GPU memory)
    device=device,  # specify the device to use (GPU or CPU)
)

# This will run until you stop it (CTRL+C), or hit "STOP" icon, or when it hits the end.

In [ ]:
dlc.evaluate_network(CONFIG_PATH, snapshots_to_evaluate=['snapshot-640'], Shuffles=[8], plotting='individual')

In [ ]:
dlc.evaluate_network(CONFIG_PATH, Shuffles=[8])

In [ ]:
absolute_video_paths = [str(Path(p).resolve()) for p in video_paths]

In [ ]:
dlc.analyze_videos(CONFIG_PATH, 
                   videos=video_paths, # if full paths (video_paths) remove destfolder parameter
                   videotype='.MOV',
                   shuffle=8,
                   trainingsetindex=0,
                   save_as_csv=True,
                  # destfolder=VIDEO_DIR, if video_paths include full paths, this is not needed, but if the videos parameter only 
                                         # has list  of video names the dest_folder is the original folder where all the videos exist 
                                         # (make it absolute path to not create new folders with same VIDEO_DIR name in it)
                   batchsize=1,
                   dynamic=(False, 0.5, 10))

In [ ]:
dlc.extract_outlier_frames(CONFIG_PATH, video_paths)  # pass a specific video

In [ ]:
%gui qt6
dlc.refine_labels(CONFIG_PATH)

In [ ]:
dlc.filterpredictions(CONFIG_PATH, video_paths)

In [ ]:
dlc.create_labeled_video(CONFIG_PATH, video_paths, shuffle=8, videotype='.MOV',  pcutoff=0.04, fastmode=True, draw_skeleton=True)

In [ ]:
%matplotlib notebook
dlc.plot_trajectories(CONFIG_PATH, video_paths, shuffle=8, videotype='.MOV', pcutoff=0.05)